## Computational Physics -- Problem Set 7
##### _Theodore Landa_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import scipy as sp
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.

_(a)_ The mass of the Sun is 1 Solar Mass, the mass of Jupiter is `9.547919e-4` Solar Masses, and the distance from Sun to Jupiter (the semi-major axis in particular) is `5.2038` a.u.; Jupiter's circular velocity $v_c$ is `2.75434` a.u. / year.

_(b)_ Code!

_(c)_ Plots! The Euler orbits don't close; nowhere close. The Leapfrog orbits are quite close to closing, though they miss by just a little bit; this might be because of the fact that there's a nonzero initial total momentum so the system translates in y by a bit every orbit.

_(d)_ Leapfrog also conserves angular momentum (unlike euler)! Which is interesting, because the symmetry baked into leapfrog is time-reversal, not rotational! Leapfrog is just goated.

In [ ]:
G = 4 * np.pi**2 # a.u.^3 yr^-2 M_sun^-1 
M_sun = 1 # Solar Masses
M_jup = 9.547919e-4
R_jup = 5.2038 # semi-major axis in a.u.
v_c = np.sqrt(G * M_sun / R_jup)
print(v_c)

In [ ]:
def accelerations(xs):
    return np.array([(xs[1]-xs[0]) * M_jup * G / np.linalg.norm(xs[0]-xs[1])**3,
                     (xs[0]-xs[1]) * M_sun * G / np.linalg.norm(xs[0]-xs[1])**3])

In [ ]:
def euler(xs_i, f, start, end, step):
    ts = np.arange(start, end, step)
    orbit = np.zeros((*xs_i.shape, ts.shape[0]))
    xs = xs_i.copy()
    for i in range(ts.shape[0]):
        orbit[...,i] = xs
        xs += step * f(xs, ts[i])
    return orbit

In [ ]:
def leapfrog(xs_i, f, start, end, step):
    ts = np.arange(start, end, step)
    orbit = np.zeros((*xs_i.shape, ts.shape[0]))
    xs = xs_i.copy()
    xs_next = euler(xs_i, f, start, start + step, step / 2)[...,-1]
    for i in range(ts.shape[0]):
        orbit[...,i] = xs
        xs += step * f(xs_next, ts[i] + step / 2)
        xs_next += step * f(xs, ts[i])
    return orbit

In [ ]:
def bigf(xs, t):
    return np.array([xs[1], accelerations(xs[0])])

xs_i = np.array([[[0, 0], [R_jup, 0]], [[0, 0], [0, 0.9 * v_c]]])

In [ ]:
euler_orbits = euler(xs_i, bigf, 0, 100, 0.01)

In [ ]:
leapfrog_orbits = leapfrog(xs_i, bigf, 0, 100, 0.01)

In [ ]:
plt.plot(euler_orbits[0, 1, 0, :], euler_orbits[0, 1, 1, :], label="Euler", c="blue")
plt.plot(leapfrog_orbits[0, 1, 0, :], leapfrog_orbits[0, 1, 1, :], label="Leapfrog", c="red")
plt.axis("equal")
plt.xlabel("distance [a.u.]")
plt.ylabel("distance [a.u.]")
plt.title("Jupiter's orbit")
plt.legend(loc="upper left")
plt.show()

In [ ]:
plt.plot(leapfrog_orbits[0, 1, 0, :], leapfrog_orbits[0, 1, 1, :], label="Leapfrog", c="red")
plt.xlim(5.1, 5.25)
plt.ylim(-0.4, 0.5)
plt.xlabel("distance [a.u.]")
plt.ylabel("distance [a.u.]")
plt.title("Jupiter's orbit close up")
plt.legend(loc="upper left")
plt.show()

In [ ]:
plt.plot(euler_orbits[0, 0, 0, :], euler_orbits[0, 0, 1, :], label="Euler", c="blue")
plt.plot(leapfrog_orbits[0, 0, 0, :], leapfrog_orbits[0, 0, 1, :], label="Leapfrog", c="red")
plt.axis("equal")
plt.xlabel("distance [a.u.]")
plt.ylabel("distance [a.u.]")
plt.title("Sun's orbit")
plt.legend(loc="upper left")
plt.show()

In [ ]:
def angular_momentum(ms, xs, vs):
    return ms[0] * (xs[0, 0,...] * vs[0, 1,...] - xs[0, 1,...] * vs[0, 0,...]) + ms[1] * (xs[1, 0,...] * vs[1, 1,...] - xs[1, 1,...] * vs[1, 0,...])

In [ ]:
Lz_euler = angular_momentum(np.array([M_sun, M_jup]), euler_orbits[0], euler_orbits[1])

In [ ]:
Lz_leapfrog = angular_momentum(np.array([M_sun, M_jup]), leapfrog_orbits[0], leapfrog_orbits[1])

In [ ]:
ts = np.arange(0, 100, 0.01)
plt.plot(ts, Lz_euler, label="Euler", c="blue")
plt.plot(ts, Lz_leapfrog, label="Leapfrog", c="red")
plt.title("Angular Momentum over time")
plt.xlabel("time [yrs]")
plt.ylabel("angular momentum [a.u.^2 M_sun / yr]")
plt.legend(loc="lower right")
plt.show()

### Problem 2.

_(a)_ Code!

_(b)_ Plot!

In [ ]:
sigma = 10
r = 28
b = 8/3
xs_i_lorenz = np.array([0., 1., 0.])

In [ ]:
def lorenz(t, xs):
    x, y, z = xs[0], xs[1], xs[2]
    return np.array([sigma * (y - x), r * x - y - x * z, x * y - b * z])

In [ ]:
lorenz_sol = sp.integrate.solve_ivp(lorenz, [0, 50], xs_i_lorenz, max_step=0.001)

In [ ]:
plt.plot(lorenz_sol.y[0], lorenz_sol.y[2], "b")
plt.title("Projection of a Lorenz orbit onto the xz-plane")
plt.xlabel("x")
plt.ylabel("z")
plt.show()

### Problem 3.

_(a)_ Plot!

_(b)_ Plot! My resonance frequency is about 9.5 s^-1.

In [ ]:
C = 2
Omega = 5
Omega_res = 9.5
g = 9.81
l = 0.1
theta_i = np.array([0., 0.])

In [ ]:
def pend(theta, t):
    return np.array([theta[1], 0-(g/l)*np.sin(theta[0])+C*np.cos(theta[0])*np.sin(Omega*t)])

def pend_res(theta, t):
    return np.array([theta[1], 0-(g/l)*np.sin(theta[0])+C*np.cos(theta[0])*np.sin(Omega_res*t)])

In [ ]:
pend_orbit = leapfrog(theta_i, pend, 0, 100, 0.01)

In [ ]:
ts_pend = np.arange(0, 100, 0.01)
plt.plot(ts_pend, pend_orbit[0], 'r')
plt.xlabel("time t [s]")
plt.ylabel("angle theta [rad]")
plt.title("Driven pendulum theta vs t for Omega = 2")
plt.show()

In [ ]:
pend_res_orbit = leapfrog(theta_i, pend_res, 0, 100, 0.01)
plt.plot(ts_pend, pend_res_orbit[0], 'r')
plt.xlabel("time t [s]")
plt.ylabel("angle theta [rad]")
plt.title("Driven pendulum theta vs t for Omega = 9.5 (at resonance)")
plt.show()

### Problem 4.

_(a)_ We are given the kinetic energy $$T=m\,\ell^2\,\left[\dot{\theta}_1^2+\frac{1}{2}\,\dot{\theta}_2^2+\dot{\theta}_1\,\dot{\theta}_2\,\cos(\theta_1-\theta_2)\right]$$ and the potential energy $$V=-m\,g\,\ell\,(2\,\cos\theta_1+\cos\theta_2)$$Combining these to get the total mechanical energy $E$, and using the $\omega$ notation, we have $$E=m\,\ell^2\,\left[\omega_1^2+\frac{1}{2}\,\omega_2^2+\omega_1\,\omega_2\,\cos(\theta_1-\theta_2)\right]-m\,g\,\ell\,(2\,\cos\theta_1+\cos\theta_2)$$

_(b)_ Plot!

_(c)_ Check out the video attached to this submission.

In [ ]:
xs_i_double_pend = np.array([np.pi / 2, np.pi / 2, 0, 0])
l_double = 0.4

In [ ]:
def double_pend(t, xs):
    th1, th2, w1, w2 = xs[0], xs[1], xs[2], xs[3]
    w1dot = 0-(w1**2*np.sin(2*th1-2*th2)+2*w2**2*np.sin(th1-th2)+(g/l_double)*(np.sin(th1-2*th2)+3*np.sin(th1)))/(3-np.cos(2*th1-2*th2))
    w2dot = (4*w1**2*np.sin(th1-th2)+w2**2*np.sin(2*th1-2*th2)+2*(g/l_double)*(np.sin(2*th1-th2)-np.sin(th2)))/(3-np.cos(2*th1-2*th2))
    return np.array([w1, w2, w1dot, w2dot])

In [ ]:
double_pend_sol = sp.integrate.solve_ivp(double_pend, [0, 100], xs_i_double_pend, max_step=0.001)

In [ ]:
def energy(m, sol):
    return m*l_double**2*(sol[2]**2+0.5*sol[3]**2+sol[2]*sol[3]*np.cos(sol[0]-sol[1]))-m*g*l_double*(2*np.cos(sol[0])+np.cos(sol[1]))

In [ ]:
energy_sol = energy(1, double_pend_sol.y)
plt.plot(double_pend_sol.t, energy_sol, 'r')
plt.xlabel("time t [s]")
plt.ylabel("total energy E [J]")
plt.title("Double pendulum total energy E vs t")
plt.show()

In [ ]:
y1_double = 0-l_double*np.cos(double_pend_sol.y[0])
ys_double = np.array([np.zeros(y1_double.shape[0]), y1_double, y1_double-l_double*np.cos(double_pend_sol.y[1])]).T
x1_double = l_double*np.sin(double_pend_sol.y[0])
xs_double = np.array([np.zeros(x1_double.shape[0]), x1_double, x1_double+l_double*np.sin(double_pend_sol.y[1])]).T

fig, ax = plt.subplots()
ax.set(xlim=(-1, 1), ylim=(-1, 1), xlabel="x [m]", ylabel="y [m]", title="Double pendulum animation!")
artist = ax.plot(xs_double[0], ys_double[0], "-ob")[0]

def update(frame):
    artist.set_xdata(xs_double[frame])
    artist.set_ydata(ys_double[frame])
    return artist,

ani = animation.FuncAnimation(fig=fig, func=update, frames=np.arange(double_pend_sol.t.size)[::20], interval=20, blit=True)
plt.show()

In [ ]:
#ani.save(filename="Pset7_double_pend.mp4", writer="ffmpeg", fps=50)

### Credits
The matplotlib documentation, the scipy documentation, wikipedia articles "Jupiter," "Solar Mass," and "Gravitational Constant," Gabe Rosen-Turits for confirming that my graphs look right, Claude for helping me debug (apparently I don't know how to save an array properly)!